In [3]:
import pandas as pd
from energy_billing import izvrši_obracun

In [12]:
net_system_cost=11250
annual_maintenance_cost=0.02*net_system_cost
degradation_rate=0.8
annual_rate_increase=2  
inflation_rate=2
discount_rate=4
consumption_df = pd.read_csv('Load_data.csv')
prices_df = pd.read_csv('Bijeli (VT_NT).csv')
prices_csv='Bijeli (VT_NT).csv'
production_data = pd.read_csv('PV.csv')
no_pv_df=production_data.copy()
no_pv_df['ac_power_kw']=0

In [13]:
years = list(range(1, 25+1))
yearly_production = []
yearly_maintenance = []

yearly_savings_month = [-net_system_cost]
yearly_savings_15min = [-net_system_cost]

discounted_yearly_savings_month= [-net_system_cost]
discounted_yearly_savings_15min= [-net_system_cost]                    
                    
yearly_cumulative_savings_month = [-net_system_cost]
yearly_cumulative_savings_15min = [-net_system_cost]             

cumulative_savings_month = -net_system_cost  # Start with negative investment
cumulative_savings_15min = -net_system_cost  # Start with negative investment

for year in years:
    # Apply degradation to production
    year_degradation_factor = (1 - degradation_rate/100) ** (year - 1)
    year_production_data = production_data * year_degradation_factor 
    
    # Apply electricity price increase
    year_rate_factor = (1 + annual_rate_increase/100) ** (year - 1)
    ht_lt_cols = [col for col in prices_df.columns if 'HT' in col.upper() or 'LT' in col.upper()]                        

    # Multiply the selected columns by the factor
    prices_df[ht_lt_cols] = prices_df[ht_lt_cols] * year_rate_factor

    obracun_energije_month, racuni_month, bilanca_month, godišnji_month = izvrši_obracun(year_production_data,consumption_df,prices_csv,net_interval='month')
    obracun_energije_15min, racuni_15min, bilanca_15min,godišnji_15min = izvrši_obracun(year_production_data,consumption_df,prices_csv,net_interval='15min')
    obracun_energije_noPV, racuni_noPV, bilanca_noPV,godišnji_noPV = izvrši_obracun(no_pv_df,consumption_df,prices_csv)

    total_month = bilanca_month.loc['Year', 'Bill']
    total_15min = bilanca_15min.loc['Year', 'Bill']
    total_noPV = bilanca_noPV.loc['Year', 'Bill']

    # Calculate baseline cost with rate increase
    year_baseline_cost = total_noPV
    # Calculate maintenance with inflation
    year_maintenance = annual_maintenance_cost * (1 + inflation_rate/100) ** (year - 1)  # Assume 2% inflation 
    # Calculate savings
    year_savings_month = year_baseline_cost-year_maintenance-total_month
    year_savings_15min = year_baseline_cost-year_maintenance-total_15min

    discounted_year_savings_month = year_savings_month / ((1 + discount_rate/100) ** year)
    discounted_year_savings_15min = year_savings_15min / ((1 + discount_rate/100) ** year)

    # Update cumulative savings
    cumulative_savings_month += discounted_year_savings_month
    cumulative_savings_15min += discounted_year_savings_15min
                                                                                                                                            
    year_production=year_production_data['ac_power_kw'].sum()
    
    # Store values
    yearly_production.append(year_production)
    yearly_maintenance.append(year_maintenance)

    yearly_savings_15min.append(year_savings_15min)
    yearly_savings_month.append(year_savings_month)     

    discounted_yearly_savings_month.append(discounted_year_savings_month)
    discounted_yearly_savings_15min.append(discounted_year_savings_15min)         

    yearly_cumulative_savings_month.append(cumulative_savings_month)
    yearly_cumulative_savings_15min.append(cumulative_savings_15min)

    

for item in yearly_cumulative_savings_month:
    if item >= 0:
        discounted_payback_years_month = years[yearly_cumulative_savings_month.index(item)]
        break
for item in yearly_cumulative_savings_15min:
    if item >= 0:
        discounted_payback_years_15min = years[yearly_cumulative_savings_15min.index(item)]
        break

# Calculate NPV and IRR                  


# NPV calculation
npv_month = sum(discounted_yearly_savings_month)
npv_15min =sum(yearly_cumulative_savings_15min)

AttributeError: 'RangeIndex' object has no attribute 'month'